# AdaGrad
The elongated bowl of a sparse feature, gradient descent / momentum / AdaGrad on it, AdaGrad's shrinking
learning rates, and AdaGrad against SGD on movie reviews (a sparse, bag-of-words dataset).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## The students data: IIT (1 for 10 of 100 students) and the package in lakh rupees
Made-up data, created once with: iit = random < 0.1, package = 3 + 6 iit + normal noise (sd 0.5).

In [ ]:
d = pd.read_csv("data/students.csv")
X = np.c_[d.iit, np.ones(len(d))]          # column for m (IIT) and column for b (always 1)
y = d.package.to_numpy()
best = np.linalg.lstsq(X, y, rcond=None)[0]
print("students:", len(d), " IIT = 1:", d.iit.sum(), " best (m, b):", best.round(2))
loss = lambda p: np.mean((y - X @ p) ** 2)
grad = lambda p: -2 * X.T @ (y - X @ p) / len(y)
start = np.array([-4.0, -4.0])
print("gradient at the start (dL/dm, dL/db):", grad(start).round(2))
print("rows where IIT = 0 add nothing to dL/dm:", int((d.iit == 0).sum()), "of", len(d))

## Three optimizers from (m, b) = (-4, -4)

In [ ]:
def run(kind, eta, steps=500, beta=0.9):
    p, v, s = start.copy(), np.zeros(2), np.zeros(2)
    path = [p.copy()]
    for _ in range(steps):
        g = grad(p)
        if kind == "gd":
            p = p - eta * g
        elif kind == "momentum":
            v = beta * v + eta * g
            p = p - v
        else:                                      # AdaGrad
            s = s + g ** 2
            p = p - eta * g / (np.sqrt(s) + 1e-8)
        path.append(p.copy())
    return np.array(path)

def steps_to(P, tol=0.01):
    gap = np.array([loss(p) for p in P]) - loss(best)
    return int(np.argmax(gap < tol)) if (gap < tol).any() else None

for kind, eta in (("gd", 0.3), ("momentum", 0.1), ("adagrad", 2.0), ("adagrad", 0.5)):
    P = run(kind, eta)
    print(f"{kind:9s} eta {eta}: loss within 0.01 of the best after {steps_to(P)} steps; "
          f"(m, b) after 10 steps {P[10].round(2)}; distance to the best after 500 steps {np.linalg.norm(P[-1] - best):.3g}")
P = run("momentum", 0.1)
print("momentum's largest b:", P[:, 1].max().round(2), "(best b is", best[1].round(2), ")")

## AdaGrad's effective learning rates eta / sqrt(v), first step by hand and over 500 steps

In [ ]:
g = grad(start)
v = g ** 2
print("v1:", v.round(1), " eta/sqrt(v1) with eta = 2:", (2 / np.sqrt(v)).round(3), " first move:", (2 / np.sqrt(v) * np.abs(g)).round(3))
for eta in (2.0, 0.5):
    p, s, eff = start.copy(), np.zeros(2), []
    for _ in range(500):
        g = grad(p)
        s = s + g ** 2
        eff.append(eta / np.sqrt(s))
        p = p - eta * g / (np.sqrt(s) + 1e-8)
    eff = np.array(eff)
    print(f"eta {eta}: effective rate (m, b) at step 1 {eff[0].round(3)}, step 10 {eff[9].round(3)}, "
          f"step 500 {eff[-1].round(3)}; ratio m/b at 500: {eff[-1, 0] / eff[-1, 1]:.1f}")

## Movie reviews (IMDB): a sparse dataset
Each review becomes 5,000 features: 1 if the review contains one of the 5,000 most common words, else 0.
10,000 reviews for training, 10,000 for validation. Logistic regression (one sigmoid node).

In [ ]:
V = 5000
(xa, ya), (xb, yb) = keras.datasets.imdb.load_data(num_words=V)
xa, ya, xb, yb = xa[:10000], ya[:10000], xb[:10000], yb[:10000]

def multi_hot(reviews):
    out = np.zeros((len(reviews), V), dtype="float32")
    for i, words in enumerate(reviews):
        out[i, words] = 1.0
    return out

Xa, Xb = multi_hot(xa), multi_hot(xb)
freq = Xa.mean(0)                                  # share of training reviews that contain each word
print(f"share of entries that are 0: {1 - Xa.mean():.3f};  words in fewer than 1% of reviews: {(freq < 0.01).mean():.2f}")

SGD and AdaGrad with the same learning rate 0.1, 10 epochs, batch size 64, 3 seeds.
Keras' AdaGrad starts its sums at 0.1 by default; we start them at 0, as in the formula.

In [ ]:
def train(kind, seed):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([keras.Input(shape=(V,)), keras.layers.Dense(1, activation="sigmoid")])
    opt = (keras.optimizers.SGD(learning_rate=0.1) if kind == "sgd"
           else keras.optimizers.Adagrad(learning_rate=0.1, initial_accumulator_value=0.0))
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    h = model.fit(Xa, ya, epochs=10, batch_size=64, validation_data=(Xb, yb), verbose=0).history
    return model, opt, h

bands = [(0, 0.003), (0.003, 0.01), (0.01, 0.1), (0.1, 1.01)]
summary, weights, eff_lr = [], {}, None
for kind in ("sgd", "adagrad"):
    for seed in range(3):
        model, opt, h = train(kind, seed)
        w = np.abs(model.layers[0].kernel.numpy().ravel())
        weights.setdefault(kind, []).append(w)
        summary.append(dict(optimizer=kind, seed=seed, loss=h["loss"][-1], val_accuracy=h["val_accuracy"][-1]))
        if kind == "adagrad" and seed == 0:
            acc = [v for v in opt.variables if "accumulator" in v.path][0].numpy().ravel()
            eff_lr = 0.1 / np.sqrt(acc + 1e-7)
        print(f"{kind:8s} seed {seed}: training loss {h['loss'][-1]:.4f}, validation accuracy {h['val_accuracy'][-1]:.4f}")
print(pd.DataFrame(summary).groupby("optimizer")[["loss", "val_accuracy"]].mean().round(4))

In [ ]:
words = pd.DataFrame(dict(freq=freq, eff_lr=eff_lr, w_sgd=np.mean(weights["sgd"], 0),
                          w_adagrad=np.mean(weights["adagrad"], 0)))
words.round(6).to_csv("data/imdb_words.csv", index=False)
for lo, hi in bands:
    s = words[(words.freq >= lo) & (words.freq < hi)]
    print(f"words in {lo:.1%}-{min(hi, 1):.1%} of reviews: {len(s):4d} words; AdaGrad median effective rate "
          f"{s.eff_lr.median():.3f}; mean |weight| SGD {s.w_sgd.mean():.3f}, AdaGrad {s.w_adagrad.mean():.3f}")
rare, common = words[words.freq < 0.003], words[words.freq >= 0.1]
print(f"rarest vs most common words: effective rate ratio {rare.eff_lr.median() / common.eff_lr.median():.1f}")